Product Brochure Generator with Frontier LLMs using Gradio UI

In [2]:
import os
import json
import gradio as gr
from dotenv import load_dotenv
from pathlib import Path
import sys

repo_root = next(
    (path for path in [Path.cwd(), *Path.cwd().parents] if (path / "week2" / "scraper.py").is_file()),
    None,
)

if repo_root is None:
    raise FileNotFoundError("Could not locate the repository root containing 'week2/scraper.py'.")

sys.path.insert(0, str(repo_root))

from week2.scraper import fetch_website_contents, fetch_website_links
from openai import OpenAI

In [3]:
load_dotenv(override=True)

openai_api_key = os.getenv("OPENAI_API_KEY")  
anthropic_api_key = os.getenv("ANTHROPIC_API_KEY")  
google_api_key = os.getenv("GOOGLE_API_KEY")


if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")
    
if anthropic_api_key:
    print(f"Anthropic API Key exists and begins {anthropic_api_key[:7]}")
else:
    print("Anthropic API Key not set")

if google_api_key:
    print(f"Google API Key exists and begins {google_api_key[:8]}")
else:
    print("Google API Key not set")


anthropic_url = "https://api.anthropic.com/v1/"
gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/"

MODEL = 'gpt-5-nano'

openai = OpenAI()
anthropic = OpenAI(api_key=anthropic_api_key, base_url=anthropic_url)
gemini = OpenAI(api_key=google_api_key, base_url=gemini_url)

OpenAI API Key exists and begins sk-proj-
Anthropic API Key exists and begins sk-ant-
Google API Key exists and begins AQ.Ab8RN


In [4]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [5]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [6]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [7]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [8]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

In [9]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [10]:
def stream_brochure_gpt(company_name, url):
    stream = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        yield response

In [11]:
def stream_brochure_claude(company_name, url):
    stream = anthropic.chat.completions.create(
        model="claude-sonnet-4-5-20250929",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        yield response

In [12]:
def stream_brochure_gemini(company_name, url):
    stream = gemini.chat.completions.create(
        model="gemini-3.6-flash",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        yield response

In [13]:
def stream_model(company, url, model):
    if model=="GPT":
        result = stream_brochure_gpt(company, url)
    elif model=="Claude":
        result = stream_brochure_claude(company, url)
    elif model=="Gemini":
        result = stream_brochure_gemini(company, url)
    else:
        raise ValueError("Unknown model")
    yield from result

In [ ]:
message_input = gr.Textbox(label="The company name :", info="Enter the company name for the product brochure generation by LLM", lines=7)
url_input = gr.Textbox(label="The link :", info="Enter the link for the product brochure generation by LLM", lines=7)
model_selector = gr.Dropdown(["GPT", "Claude", "Gemini"], label="Select model", value="GPT")
message_output = gr.Markdown(label="Response:")

view = gr.Interface(
    fn=stream_model, 
    inputs=[message_input, url_input, model_selector], 
    outputs=message_output, 
    title="Product Brochure Generator with Frontier LLMs",
    examples=[
            ["Hugging Face", "https://huggingface.co", "GPT"],
            ["Anthropic", "https://www.anthropic.com", "Claude"],
            ["Edward Donner", "https://edwarddonner.com", "Gemini"]
        ], 
    flagging_mode="never"
    )
view.launch(share=True)

* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://3e96660358463d9e01.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Selecting relevant links for https://edwarddonner.com by calling gpt-5-nano
Found 6 relevant links
